# Day 1 — Data Preparation

## Objective

Prepare and clean the Cardiovascular Disease dataset for exploratory data analysis and baseline modeling.

## Tasks

- Load the dataset
- Inspect the data structure
- Check missing values and duplicates
- Convert age from days to years
- Remove invalid blood pressure values
- Verify the cleaned dataset
- Save the cleaned dataset for the next notebooks

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("cardio.csv", sep=";")
df.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [3]:
print("Dataset shape:", df.shape)

Dataset shape: (70000, 13)


In [4]:
df.describe()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
count,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000,70000.000000
mean,49972.419900,19468.865814,1.349571,164.359229,74.205690,128.817286,96.630414,1.366871,1.226457,0.088129,0.053771,0.803729,0.499700
std,28851.302323,2467.251667,0.476838,8.210126,14.395757,154.011419,188.472530,0.680250,0.572270,0.283484,0.225568,0.397179,0.500003
min,0.000000,10798.000000,1.000000,55.000000,10.000000,-150.000000,-70.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,25006.750000,17664.000000,1.000000,159.000000,65.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000
50%,50001.500000,19703.000000,1.000000,165.000000,72.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000
75%,74889.250000,21327.000000,2.000000,170.000000,82.000000,140.000000,90.000000,2.000000,1.000000,0.000000,0.000000,1.000000,1.000000
max,99999.000000,23713.000000,2.000000,250.000000,200.000000,16020.000000,11000.000000,3.000000,3.000000,1.000000,1.000000,1.000000,1.000000


In [5]:
df.isnull().sum()

id             0
age            0
gender         0
height         0
weight         0
ap_hi          0
ap_lo          0
cholesterol    0
gluc           0
smoke          0
alco           0
active         0
cardio         0
dtype: int64

In [6]:
print("Total missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Total missing values: 0
Duplicate rows: 0


### Age Conversion

The original age feature is recorded in days. It is converted to years using 365.25 days per year.

In [7]:
df["age_years"] = df["age"] / 365.25
df.drop(columns=["age"], inplace=True)

In [8]:
df[["age_years"]].describe()

,age_years
count,70000.000000
mean,53.302850
std,6.754967
min,29.563313
25%,48.361396
50%,53.943874
75%,58.390144
max,64.922656


In [9]:
print("ap_hi range:", df["ap_hi"].min(), "to", df["ap_hi"].max())
print("ap_lo range:", df["ap_lo"].min(), "to", df["ap_lo"].max())

ap_hi range: -150 to 16020
ap_lo range: -70 to 11000


### Blood Pressure Cleaning

The ap_hi and ap_lo features contain invalid and unrealistic values.

To remove clearly invalid measurements while retaining possible extreme blood pressure values, the following ranges are used:

- ap_hi: 60–350
- ap_lo: 40–250

In [10]:
def clean_blood_pressure(df):
    valid_bp = (
        df["ap_hi"].between(60, 350) &
        df["ap_lo"].between(40, 250)
    )

    df.drop(index=df.index[~valid_bp], inplace=True)

    return df


clean_blood_pressure(df)

,id,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years
0,0,2,168,62.0,110,80,1,1,0,0,1,0,50.357290
1,1,1,156,85.0,140,90,3,1,0,0,1,1,55.381246
2,2,1,165,64.0,130,70,3,1,0,0,0,1,51.627652
3,3,2,169,82.0,150,100,1,1,0,0,1,1,48.249144
4,4,1,156,56.0,100,60,1,1,0,0,0,0,47.841205
...,...,...,...,...,...,...,...,...,...,...,...,...,...
69995,99993,2,168,76.0,120,80,1,1,1,0,1,0,52.676249
69996,99995,1,158,126.0,140,90,2,2,0,0,1,1,61.878166
69997,99996,2,183,105.0,180,90,3,1,0,1,0,1,52.199863
69998,99998,1,163,72.0,135,80,1,2,0,0,0,1,61.412731


In [11]:
print("ap_hi range:", df["ap_hi"].min(), "-", df["ap_hi"].max())
print("ap_lo range:", df["ap_lo"].min(), "-", df["ap_lo"].max())

ap_hi range: 60 - 240
ap_lo range: 40 - 190


### Height Cleaning

The height feature contains unrealistic values that may result from data entry errors.

To remove clearly invalid measurements, height values outside the range of 100–220 cm are excluded.

In [12]:
def clean_height(df):
    invalid_height = ~df["height"].between(100, 220)

    df.drop(index=df.index[invalid_height], inplace=True)

    return df


clean_height(df)

,id,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years
0,0,2,168,62.0,110,80,1,1,0,0,1,0,50.357290
1,1,1,156,85.0,140,90,3,1,0,0,1,1,55.381246
2,2,1,165,64.0,130,70,3,1,0,0,0,1,51.627652
3,3,2,169,82.0,150,100,1,1,0,0,1,1,48.249144
4,4,1,156,56.0,100,60,1,1,0,0,0,0,47.841205
...,...,...,...,...,...,...,...,...,...,...,...,...,...
69995,99993,2,168,76.0,120,80,1,1,1,0,1,0,52.676249
69996,99995,1,158,126.0,140,90,2,2,0,0,1,1,61.878166
69997,99996,2,183,105.0,180,90,3,1,0,1,0,1,52.199863
69998,99998,1,163,72.0,135,80,1,2,0,0,0,1,61.412731


### Weight Cleaning

The weight feature contains unrealistic values that may result from data entry errors.

To remove clearly invalid measurements, weight values outside the range of 30–200 kg are excluded.

In [13]:
def clean_weight(df):
    invalid_weight = ~df["weight"].between(30, 200)

    df.drop(index=df.index[invalid_weight], inplace=True)

    return df


clean_weight(df)

,id,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years
0,0,2,168,62.0,110,80,1,1,0,0,1,0,50.357290
1,1,1,156,85.0,140,90,3,1,0,0,1,1,55.381246
2,2,1,165,64.0,130,70,3,1,0,0,0,1,51.627652
3,3,2,169,82.0,150,100,1,1,0,0,1,1,48.249144
4,4,1,156,56.0,100,60,1,1,0,0,0,0,47.841205
...,...,...,...,...,...,...,...,...,...,...,...,...,...
69995,99993,2,168,76.0,120,80,1,1,1,0,1,0,52.676249
69996,99995,1,158,126.0,140,90,2,2,0,0,1,1,61.878166
69997,99996,2,183,105.0,180,90,3,1,0,1,0,1,52.199863
69998,99998,1,163,72.0,135,80,1,2,0,0,0,1,61.412731


## Final Validation

After completing the data cleaning steps, the dataset is checked again to ensure that:

- No missing values remain.
- No duplicate rows remain.
- All cleaned features are within the defined valid ranges.
- The final dataset is ready for exploratory data analysis and baseline modeling.

In [14]:
print("Final dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nFeature ranges:")
print("Age:", round(df["age_years"].min(), 2), "-", round(df["age_years"].max(), 2))
print("Height:", df["height"].min(), "-", df["height"].max())
print("Weight:", df["weight"].min(), "-", df["weight"].max())
print("ap_hi:", df["ap_hi"].min(), "-", df["ap_hi"].max())
print("ap_lo:", df["ap_lo"].min(), "-", df["ap_lo"].max())

Final dataset shape: (68742, 13)
Missing values: 0
Duplicate rows: 0

Feature ranges:
Age: 29.56 - 64.92
Height: 100 - 207
Weight: 30.0 - 200.0
ap_hi: 60 - 240
ap_lo: 40 - 190


## Save Cleaned Dataset

The cleaned dataset is saved as a CSV file to be used in the following notebooks.

In [15]:
df.to_csv("cardio_cleaned.csv", index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
